In [17]:
import pandas as pd
import numpy as np
import numpy.ma as ma

df = pd.read_csv("sp500_companies.csv")
sector_dict = dict(zip(df.Symbol, df.Sector))


def comput_log_returns(prices_df: pd.DataFrame):
    return np.log(prices_df / prices_df.shift(1)).dropna().values


def compute_covariances(X: pd.DataFrame):
    return np.cov(X.T)  # We want num_Stocks x num_Stocks shaped


def _load_covariance(universe_id: str):
    """Returns (A, tickers) , covariance matrix + list of ticker symbols in same order as A's rows/columns

    Args:
        universe_id (str): ie) "sp500" for sp500
    """
    df = pd.read_csv(
        "sp500_prices.csv",
        index_col="Date",
        parse_dates=True,
    )

    log_return = comput_log_returns(df)

    cov = compute_covariances(log_return)

    return cov, df.columns.tolist()

cov, tickers = _load_covariance("sp500")

In [28]:
from grad_fw import FWHomotopySolver

solver = FWHomotopySolver(cov, k=20, n_steps=800, n_mc_samples=100)

solution = solver.solve()


[FWHomotopy] p=472, k=20, steps=800, n_mc=100, alpha=0.01


In [30]:
# fw_indices = ma.masked_array(tickers, mask=solution)
fw_indices = ma.masked_where(solution < 0.5, tickers).compressed()
list(zip(fw_indices, [1/20] * 20))

[(np.str_('AMD'), 0.05),
 (np.str_('APA'), 0.05),
 (np.str_('CCL'), 0.05),
 (np.str_('CZR'), 0.05),
 (np.str_('DVN'), 0.05),
 (np.str_('ENPH'), 0.05),
 (np.str_('HAL'), 0.05),
 (np.str_('IVZ'), 0.05),
 (np.str_('LRCX'), 0.05),
 (np.str_('MGM'), 0.05),
 (np.str_('MPWR'), 0.05),
 (np.str_('NCLH'), 0.05),
 (np.str_('NVDA'), 0.05),
 (np.str_('ON'), 0.05),
 (np.str_('OXY'), 0.05),
 (np.str_('PCG'), 0.05),
 (np.str_('SMCI'), 0.05),
 (np.str_('TRGP'), 0.05),
 (np.str_('TSLA'), 0.05),
 (np.str_('UAL'), 0.05)]